# Room number annotation for iPhone LiDAR exports — local Qwen test

This test copy runs Qwen3-VL-2B-Instruct inside Colab. Upload a ZIP named `AppFileOutput.zip` to the My Drive > Colab Notebooks; the notebook extracts it into Colab automatically. It sends no images to the Gemini API and does not need a Gemini API key.


## 1. Set Colab to use a GPU

In Colab, choose **Runtime → Change runtime type → GPU** if a GPU is available. Then run the install cell below. After installation, restart the runtime if Colab asks, then run cells from the top.


In [6]:
%pip install -q git+https://github.com/huggingface/transformers accelerate qwen-vl-utils pillow-heif


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Mount Drive and configure paths

Upload your compressed data folder to My Drive as `AppFileOutput.zip`. Run this cell to mount Drive and extract the ZIP to Colab local storage. The output is saved separately in Drive so it will not mix with Gemini results.


In [7]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import zipfile

import torch
from PIL import Image
from pillow_heif import register_heif_opener
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

register_heif_opener()

# If this is a fresh Colab session, run this cell once to mount Drive.
from google.colab import drive
drive.mount("/content/drive")

# Upload this ZIP to the top level of My Drive before running this cell.
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/Colab Notebooks/appfilesfr.zip")
EXTRACT_DIR = Path("/content/room_data")
if not DRIVE_ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Upload AppFileOutput.zip to the top level of My Drive first. Expected: {DRIVE_ZIP_PATH}"
    )
if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP_PATH) as archive:
        root = EXTRACT_DIR.resolve()
        for member in archive.infolist():
            target = (EXTRACT_DIR / member.filename).resolve()
            try:
                target.relative_to(root)
            except ValueError:
                raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
        archive.extractall(EXTRACT_DIR)
    print(f"Extracted {DRIVE_ZIP_PATH.name} to {EXTRACT_DIR}")
else:
    print(f"Using existing extracted data at {EXTRACT_DIR}. Restart the Colab runtime to extract an updated ZIP.")

# rglob in the scan cell will find images even if the ZIP contains a top-level folder.
INPUT_DIR = EXTRACT_DIR.resolve()
OUTPUT_FILE = Path("/content/drive/MyDrive/Colab Notebooks/room_annotations_qwen.jsonl")
MANIFEST_FILE = None  # Optional CSV or JSONL manifest
MODEL = "Qwen/Qwen3-VL-2B-Instruct"
RESUME = True
TEST_IMAGE_LIMIT = None  # Set to None after the test to process every discovered image

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".heic", ".webp", ".tif", ".tiff", ".bmp", ".jfif"}
PROMPT = """Inspect this image for room or door numbers (for example 204, 12A, B-17).
Return every clearly visible room-number sign, prioritizing signs that identify
a room rather than unrelated numbers. Transcribe exactly; do not infer obscured
characters. For each, give a bounding box around the visible number/sign as
[ymin, xmin, ymax, xmax], normalized to integers 0..1000. If none is legible,
return an empty list. Include a short evidence note and confidence from 0 to 1.
Return only valid JSON in this exact shape: {\"room_numbers\":[{\"text\":\"204\",\"box_2d\":[100,100,200,200],\"confidence\":0.9,\"evidence\":\"sign above door\"}]} ."""

def load_manifest(path):
    if path is None:
        return {}
    path = Path(path)
    if path.suffix.lower() == ".csv":
        with path.open(newline="", encoding="utf-8-sig") as f:
            rows = list(csv.DictReader(f))
    else:
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    result = {}
    for row in rows:
        key = row.get("image_path") or row.get("image") or row.get("image_id")
        if not key:
            raise ValueError("Each manifest record needs image_path (or image/image_id)")
        key = Path(str(key)).as_posix()
        if key in result:
            raise ValueError(f"Duplicate manifest image key: {key}")
        result[key] = row
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_qwen():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU is active. In Colab choose Runtime > Change runtime type > GPU, then rerun from the top.")
    print("GPU:", torch.cuda.get_device_name(0))
    print("Loading Qwen weights (first run downloads several GB; later runs may use the cache)...")
    processor = AutoProcessor.from_pretrained(MODEL)
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL, dtype="auto", device_map="auto"
    )
    model.eval()
    print("Qwen is ready.")
    return model, processor

def annotate_image(model, processor, path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": Path(path).resolve().as_uri()},
            {"type": "text", "text": PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    generated = generated[:, inputs.input_ids.shape[1]:]
    answer = processor.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    start = answer.find("{")
    if start < 0:
        raise ValueError(f"Model did not return JSON: {answer[:300]}")
    result, _ = json.JSONDecoder().raw_decode(answer[start:])
    numbers = result.get("room_numbers")
    if not isinstance(numbers, list):
        raise ValueError("Model response lacks a room_numbers list")
    cleaned = []
    for item in numbers:
        if not isinstance(item, dict):
            continue
        box = item.get("box_2d")
        if not isinstance(box, list) or len(box) != 4:
            continue
        box = [max(0, min(1000, int(round(float(v))))) for v in box]
        cleaned.append({
            "text": str(item.get("text", "")).strip(),
            "box_2d": box,
            "confidence": max(0.0, min(1.0, float(item.get("confidence", 0)))),
            "evidence": str(item.get("evidence", "")),
        })
    return cleaned

def lidar_siblings(path, root):
    names = {"depth_path": "depth.bin", "confidence_path": "confidence.bin", "metadata_path": "metadata.json"}
    result = {}
    for key, name in names.items():
        sibling = path.parent / name
        if sibling.is_file():
            result[key] = sibling.relative_to(root).as_posix()
    return result


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using existing extracted data at /content/room_data. Restart the Colab runtime to extract an updated ZIP.


## 3. Find every image recursively and inspect the folder count


In [8]:
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Set INPUT_DIR to your Drive export folder; not found: {INPUT_DIR}")
all_files = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file())
images = sorted(p for p in all_files if p.suffix.lower() in IMAGE_EXTENSIONS)
manifest = load_manifest(MANIFEST_FILE)
ext_counts = Counter(p.suffix.lower() or "[no extension]" for p in all_files)
print(f"Folder: {INPUT_DIR}")
print(f"All files found recursively: {len(all_files)}")
print(f"Images selected for annotation: {len(images)}")
print("File counts by extension:", dict(sorted(ext_counts.items())))
ignored_candidates = [p for p in all_files if p.suffix.lower() in {".avif", ".gif", ".raw", ".dng"}]
if ignored_candidates:
    print(f"Potential image files not selected: {len(ignored_candidates)}")
    for p in ignored_candidates[:20]:
        print("  not selected:", p.relative_to(INPUT_DIR).as_posix())
print("First 10 image paths:")
for p in images[:10]:
    print(" ", p.relative_to(INPUT_DIR).as_posix())
if len(images) > 10:
    print(f"... {len(images) - 10} additional images are included; this is not a 10-image limit.")


Folder: /content/room_data
All files found recursively: 8587
Images selected for annotation: 1908
File counts by extension: {'.bin': 3816, '.jpg': 1908, '.json': 1908, '[no extension]': 955}
First 10 image paths:
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000002/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000003/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000004/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000005/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000032/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000033/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000034/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000035/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000050/rgb.jpg
  9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000051/rgb.jpg
... 1898 additional images are included; this is not a 10-image limit.


The scan uses `rglob`, so it searches all subfolders; the 10 paths above are only a preview. Images with the listed extensions are all queued. The extension summary helps identify image formats the scanner may have skipped.


## 4. Run local Qwen annotation

The first run downloads the Qwen model into the Colab runtime and may take several minutes. The first run processes only five images so you can test the local model. Set `TEST_IMAGE_LIMIT = None` in the configuration cell to process all discovered images. Each image is processed on the runtime GPU. `RESUME = True` skips successful records in the separate Qwen output file and retries records that contain an error.


In [9]:
model, processor = load_qwen()
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
done = set()
if RESUME and OUTPUT_FILE.exists():
    with OUTPUT_FILE.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if "error" not in record:
                    done.add(record["image_id"])
mode = "a" if RESUME else "w"
failed = []
images_to_process = images if TEST_IMAGE_LIMIT is None else images[:TEST_IMAGE_LIMIT]
print(f"Images queued this run: {len(images_to_process)} of {len(images)} discovered")
with OUTPUT_FILE.open(mode, encoding="utf-8") as out:
    for i, path in enumerate(images_to_process, 1):
        image_id = path.relative_to(INPUT_DIR).as_posix()
        if image_id in done:
            continue
        try:
            numbers = annotate_image(model, processor, path)
            record = {
                "image_id": image_id,
                "image_path": image_id,
                "image_sha256": sha256_file(path),
                "model": MODEL,
                "room_numbers": numbers,
                "lidar": {**lidar_siblings(path, INPUT_DIR), **manifest.get(image_id, {})},
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            print(f"[{i}/{len(images_to_process)}] {image_id}: {len(numbers)} room number(s)")
        except Exception as exc:
            record = {
                "image_id": image_id,
                "image_path": image_id,
                "image_sha256": sha256_file(path),
                "model": MODEL,
                "room_numbers": [],
                "lidar": {**lidar_siblings(path, INPUT_DIR), **manifest.get(image_id, {})},
                "error": str(exc),
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            failed.append(image_id)
            print(f"Failed: {image_id}: {exc}")
print(f"\nSaved annotations to {OUTPUT_FILE}")
print(f"Failed images this run: {len(failed)}")


GPU: NVIDIA A100-SXM4-40GB
Loading Qwen weights (first run downloads several GB; later runs may use the cache)...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Qwen is ready.
Images queued this run: 1908 of 1908 discovered
[6/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000033/rgb.jpg: 1 room number(s)
[7/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000034/rgb.jpg: 1 room number(s)
[8/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000035/rgb.jpg: 1 room number(s)
[9/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000050/rgb.jpg: 1 room number(s)
[10/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000051/rgb.jpg: 1 room number(s)
[11/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000056/rgb.jpg: 1 room number(s)
[12/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000057/rgb.jpg: 1 room number(s)
[13/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000058/rgb.jpg: 1 room number(s)
[14/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000059/rgb.jpg: 1 room number(s)
[15/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000060/rgb.jpg: 1 room number(s)
[16/1908] 9D6D7A6E-4CD6-4883-9C04-9DDDB12982EC/frame_000061/rgb.jpg: 1 room numbe

## 5. Inspect results

Each `box_2d` is `[ymin, xmin, ymax, xmax]` normalized to `0..1000`. It describes a region in the RGB image. Mapping it to depth pixels is valid only when RGB and depth are aligned; unregistered point-cloud data needs camera calibration and projection.


In [15]:
import json
import re
from collections import defaultdict

with OUTPUT_FILE.open(encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

room_frames = defaultdict(set)

for record in results:
    if "error" in record:
        continue

    image_path = record.get("image_path") or record.get("image_id") or ""
    frame_match = re.search(r"frame_(\d+)", image_path)
    if not frame_match:
        continue

    frame_number = int(frame_match.group(1))

    for room in record.get("room_numbers") or []:
        text = str(room.get("text") or "").strip().upper()
        if re.fullmatch(r"2[A-Z][0-9]{2}", text):
            room_frames[text].add(frame_number)

for room_number, frames in sorted(room_frames.items()):
    print(f"{room_number}: frames {', '.join(map(str, sorted(frames)))}")

2A01: frames 1200, 1201, 1206, 1208, 1209, 1230, 1231, 1237, 1239, 1252, 1253, 1254, 1262, 1265, 1290, 1291, 1296, 1299, 1310, 1328, 1342
2B06: frames 637, 654, 662, 663, 1917
2B08: frames 1916
2B10: frames 2280, 2287, 2300, 2330, 2331
2B12: frames 2686
